<!-- TARGET: public course-materials repo, sessions/m04/lab-solution.ipynb. Copy to work/m04/ to run it. -->
# Module 4 · Lab

**Python for Data Science · ESADE MIBA · Wed 4 Nov 2026**

Solution. Two more Olist tables, customers and products: inspect, type, filter, find what is missing.

Fill in each cell with `...`, then run the **check** cell below it. Silence means correct; an
`AssertionError` tells you what to look at.

In [ ]:
from pathlib import Path

import pandas as pd

RAW = Path("../../data/raw")

## Part 0 · Python on real rows (10 min)

Twenty real item lines from Olist, turned into a list of dictionaries, like the warm-up.

In [ ]:
items = pd.read_csv(RAW / "olist_order_items_dataset.csv").head(20).to_dict("records")
items[0]

**0a.** With a loop and an accumulator, compute `items_revenue`, the sum of `price`.

In [ ]:
items_revenue = 0
for item in items:
    items_revenue = items_revenue + item["price"]

In [ ]:
assert round(items_revenue, 2) == 3048.12, "sum price over all 20 items"

**0b.** Count the items whose `freight_value` is above 20 BRL into `n_expensive_freight`.

In [ ]:
n_expensive_freight = 0
for item in items:
    if item["freight_value"] > 20:
        n_expensive_freight = n_expensive_freight + 1

In [ ]:
assert n_expensive_freight == 3

**0c.** Write `item_total(item)` returning price plus freight. Then compute
`items_total_paid` by summing `item_total` over all items.

In [ ]:
def item_total(item):
    return item["price"] + item["freight_value"]


items_total_paid = sum(item_total(item) for item in items)

In [ ]:
assert round(item_total(items[0]), 2) == 72.19
assert round(items_total_paid, 2) == 3397.98

**0d. Stretch.** Build `revenue_by_seller`, a dictionary from `seller_id` to the sum of `price`,
by hand. Then compute the same with pandas (`groupby`) on the same 20 rows and check that the
two agree.

In [ ]:
revenue_by_seller = {}
for item in items:
    seller = item["seller_id"]
    revenue_by_seller[seller] = revenue_by_seller.get(seller, 0) + item["price"]

revenue_by_seller_pandas = pd.DataFrame(items).groupby("seller_id")["price"].sum().to_dict()

In [ ]:
assert len(revenue_by_seller) == 19, "one key per distinct seller"
assert all(round(revenue_by_seller[seller] - revenue_by_seller_pandas[seller], 6) == 0 for seller in revenue_by_seller)

## Part 1 · Customers (15 min)

**1a.** Load `olist_customers_dataset.csv` into `customers`. Look at it with `shape`, `head()`
and `dtypes`.

In [ ]:
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
customers.dtypes

In [ ]:
assert customers.shape == (99441, 5)

**1b.** `customer_zip_code_prefix` was read as a number. Brazilian zip prefixes have five
digits, and many start with 0. How many prefixes are below 10000, i.e. lost their leading zero?
Store it in `n_lost_zero`. Then reload the file so that this column stays text (hint: the `dtype`
parameter of `read_csv` takes a dictionary).

In [ ]:
n_lost_zero = (customers["customer_zip_code_prefix"] < 10000).sum()
customers = pd.read_csv(RAW / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})

In [ ]:
assert n_lost_zero == 23995
assert customers["customer_zip_code_prefix"].str.len().eq(5).all(), "every prefix should be 5 characters of text"

**1c.** How many customers live in São Paulo state (`"SP"`)? How many in Rio de Janeiro or Minas
Gerais together? Use a boolean mask and `isin`.

In [ ]:
n_sp = (customers["customer_state"] == "SP").sum()
n_rj_mg = customers["customer_state"].isin(["RJ", "MG"]).sum()

In [ ]:
assert n_sp == 41746
assert n_rj_mg == 24487

**1d.** `customer_id` has 99,441 distinct values, `customer_unique_id` fewer. Compute
`n_people`, the number of distinct `customer_unique_id`. In one sentence in the markdown cell
below: what could explain the difference? (We come back to this next week.)

In [ ]:
n_people = customers["customer_unique_id"].nunique()

In [ ]:
assert n_people == 96096

*Your explanation:* `customer_id` is created per order, so a person who buys twice gets two `customer_id`s but keeps one `customer_unique_id`. 99,441 orders, 96,096 people.

## Part 2 · Products (20 min)

**2a.** Load `olist_products_dataset.csv` into `products`. Two column names have a typo
(`lenght`). Fix them with `rename` and a dictionary.

In [ ]:
products = pd.read_csv(RAW / "olist_products_dataset.csv").rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    }
)

In [ ]:
assert products.shape == (32951, 9)
assert "product_name_length" in products.columns and "product_description_length" in products.columns

**2b.** Count missing values per column into `missing_per_column` (a Series). How many products
have no category? Are the products without a category the same ones missing the name and
photo columns? Check it rather than eyeballing.

In [ ]:
missing_per_column = products.isna().sum()
n_no_category = missing_per_column["product_category_name"]
no_category = products["product_category_name"].isna()
same_rows = (no_category == products["product_photos_qty"].isna()).all()

In [ ]:
assert n_no_category == 610
assert same_rows, "compare the two isna() masks"

**2c.** Add a derived column `volume_cm3` (length × height × width), then count the products
heavier than 10 kg into `n_heavy`. `product_weight_g` is in grams.

In [ ]:
products = products.assign(
    volume_cm3=products["product_length_cm"] * products["product_height_cm"] * products["product_width_cm"]
)
n_heavy = (products["product_weight_g"] > 10_000).sum()

In [ ]:
assert products["volume_cm3"].median() == 6840
assert n_heavy == 1891

**2d.** Show the five heaviest products with `sort_values`. Then: 4 products weigh **0 g**.
Find them. Real weight, or a data problem? Write your answer below.

In [ ]:
heaviest = products.sort_values("product_weight_g", ascending=False).head(5)
zero_weight = products[products["product_weight_g"] == 0]
zero_weight

In [ ]:
assert heaviest["product_weight_g"].iloc[0] == 40425
assert len(zero_weight) == 4

*Your answer:* Almost certainly a data problem: no physical product weighs 0 g, and their dimensions are not zero. Treat 0 as missing, not as a weight.

**2e. Stretch.** Translate the categories to English. Load
`product_category_name_translation.csv`, build a dictionary Portuguese → English, and use
`Series.map(dictionary)` to create `category_en`. How many products end up without an English
name, and why is that more than 610?

In [ ]:
translation = pd.read_csv(RAW / "product_category_name_translation.csv")
portuguese_to_english = dict(
    zip(translation["product_category_name"], translation["product_category_name_english"])
)
category_en = products["product_category_name"].map(portuguese_to_english)
untranslated = products.loc[category_en.isna() & products["product_category_name"].notna(), "product_category_name"]
untranslated.value_counts()

In [ ]:
assert category_en.isna().sum() == 623, "610 without category + products whose category has no translation"

## Done?

Next week we save the typed tables to `data/clean/` and build the first join. Bring this
notebook: we reuse your `customers` and `products` code.